In [84]:
import os
if os.path.basename(os.getcwd()) == "limpieza":
    os.chdir("..")
    
import pandas as pd
import numpy as np
from utils.funciones_filtrado import tipo_nulo_unicos_x_columna

In [85]:
base_productos = pd.read_excel('productos.xlsx')
base_productos.head()

,producto_id,producto,categoria,tamaño,activo
0,P001,Americano,café,Chico,True
1,P001,Americano,Café,Grande,True
2,P002,Latte,Café,Chico,True
3,P003,Latte Vainilla,cafe,Chico,True
4,P004,Cappuccino,Café,Mediano,True


Primero pondremos en orden a los id, ya que recordemos que hay un id repetido y un id que es nulo, es decir, un producto que no tiene id, recordemos que el único id que se repite es el 'P001' y que tiene tamaño grande. Además como ya sólo nos falta un valor nulo, pues sólo lo rellenamos.

In [86]:
base_productos.loc[(base_productos['producto_id']=='P001') & (base_productos['tamaño']=='Grande'), 'producto_id'] = 'P018'
base_productos['producto_id'] = base_productos['producto_id'].fillna('P019')
#en lugar de asignar un id manualmente podriamos tener un generador de id's automatico y que nos asigne uno, sin repetición para evitar repeticiones y mantener el estandar 
base_productos.head()

,producto_id,producto,categoria,tamaño,activo
0,P001,Americano,café,Chico,True
1,P018,Americano,Café,Grande,True
2,P002,Latte,Café,Chico,True
3,P003,Latte Vainilla,cafe,Chico,True
4,P004,Cappuccino,Café,Mediano,True


Ahora vamos a corregir el problema de las columnas categoria y activo, el cual es que para un mismo valor, se escribe de manera diferente

In [87]:
# aplicamos que todas las letras sean minusculas y que no contengan espacios
base_productos['categoria'] = base_productos['categoria'].str.lower().str.strip() 
base_productos['activo'] = base_productos['activo'].astype(str).str.lower().str.strip()

#aqui aplicaría una función donde quite acentos para así tener los textos lo más limpios posibles, pero por temas de tiempos aprovecharemos una ventaja, que todas empiezan con letras ditintas
base_productos['categoria'] = np.where(base_productos['categoria'].str.startswith('c'), 'Café', base_productos['categoria'])
base_productos['categoria'] = np.where(base_productos['categoria'].str.startswith('f'), 'Frío', base_productos['categoria'])
base_productos['categoria'] = np.where(base_productos['categoria'].str.startswith('t'), 'Té', base_productos['categoria'])
base_productos['categoria'] = np.where(base_productos['categoria'].str.startswith('a'), 'Alimento', base_productos['categoria'])

base_productos['activo'] = np.where(base_productos['activo'].str.startswith('t'), 'verdadero', base_productos['activo'])
base_productos['activo'] = np.where(base_productos['activo'].str.startswith('f'), 'falso', base_productos['activo'])
base_productos['activo'] = np.where(base_productos['activo'].str.startswith('v'), 'verdadero', base_productos['activo']) #para ese caso especial donde está escrito verdadero

base_productos = base_productos.sort_values(by='producto_id')
base_productos

,producto_id,producto,categoria,tamaño,activo
0,P001,Americano,Café,Chico,verdadero
2,P002,Latte,Café,Chico,verdadero
3,P003,Latte Vainilla,Café,Chico,verdadero
4,P004,Cappuccino,Café,Mediano,verdadero
5,P005,Espresso,Café,Solo,verdadero
6,P006,Frappé Chocolate,Frío,Mediano,verdadero
7,P007,Frappe Caramelo,Frío,Mediano,verdadero
8,P008,Té Verde,Té,Chico,verdadero
9,P009,Té Negro,Té,Chico,falso
10,P010,Muffin Arándano,Alimento,Unitario,verdadero


Por último notamos que hay un producto llamado 'Americano Grande' pero la descripción de grande ya debería de ir en tamaño, y como ya tenemos un producto 'Americano' con tamaño 'Grande' que fue el producto con id repetido, por lo que no tendría sentido seguir manteniendo un producto de esta forma, por lo que se procede a su cancelación.

In [88]:
#base_inventario.drop(base_inventario[base_inventario['producto_id']=='P016'].index, inplace=True)
base_productos.drop(base_productos[base_productos['producto']=='Americano Grande'].index, inplace=True)
base_productos

,producto_id,producto,categoria,tamaño,activo
0,P001,Americano,Café,Chico,verdadero
2,P002,Latte,Café,Chico,verdadero
3,P003,Latte Vainilla,Café,Chico,verdadero
4,P004,Cappuccino,Café,Mediano,verdadero
5,P005,Espresso,Café,Solo,verdadero
6,P006,Frappé Chocolate,Frío,Mediano,verdadero
7,P007,Frappe Caramelo,Frío,Mediano,verdadero
8,P008,Té Verde,Té,Chico,verdadero
9,P009,Té Negro,Té,Chico,falso
10,P010,Muffin Arándano,Alimento,Unitario,verdadero


In [89]:
no_registros, no_columnas = base_productos.shape
print(f"Número de productos registrados: {no_registros}, número de columnas de la tabla: {no_columnas}")
inspeccion = tipo_nulo_unicos_x_columna(base_productos)
inspeccion

Número de productos registrados: 17, número de columnas de la tabla: 5


,columna,tipo de dato,cantidad de valores nulos,cantidad de valores unicos
0,producto_id,object,0,17
1,producto,object,0,16
2,categoria,object,0,4
3,tamaño,object,0,6
4,activo,object,0,2


Como podemos ver ya no tenemos valores nulos, y ya tenemos 18 id's unicos para nuestros 18 productos, además de que en activo ya tenemos 2 valores, y las categorías ya se arreglaron, porque solamente tenemos 4.

In [90]:
base_productos.to_excel('productos_limpio.xlsx', index=False)